# Price Prediction Model — Archive Fashion Items

This notebook builds a price prediction model for the top 5 most-traded archive fashion items on Grailed. We compare **Linear Regression** (baseline) vs **XGBoost** using time-series features, rolling price averages, and an **item-type classifier** derived from listing titles.

**Pipeline:**
1. Data loading & outlier removal
2. Feature engineering (time, condition, item type, rolling averages)
3. Model training (80/20 time-series split + 3-fold CV)
4. Evaluation (MAE / RMSE / CV comparison)
5. Visualization (actual vs predicted)
6. Summary table with price trend signals

**Key improvements:**
- **Item-type feature**: Classifies listings by title keywords (jacket, pants, hoodie, tee, shirt, shoes) to reduce within-keyword price variance
- **Conservative XGBoost**: `max_depth=3, n_estimators=50, min_child_weight=5` to reduce overfitting on small datasets
- **Cross-validation**: 3-fold `TimeSeriesSplit` to validate model generalization

---

In [1]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import Markdown, display

import price_model

print("Modules loaded.")

Modules loaded.


## 1. Load & Clean Data

We load `historical_sold.csv`, filter to items with ≥30 records, and remove price outliers that deviate more than 3x from the median. This prevents extreme listings (mispriced or bundled items) from skewing the model.

In [2]:
df_raw = price_model.load_data()
print(f"Raw data: {len(df_raw)} records, {df_raw['keyword'].nunique()} items")
print(f"Date range: {df_raw['sold_date'].min().strftime('%Y-%m-%d')} ~ {df_raw['sold_date'].max().strftime('%Y-%m-%d')}")
print()

# Per-item counts before cleaning
print("Records per item:")
for kw, g in df_raw.groupby("keyword"):
    print(f"  {kw:<35} {len(g):>3} records, median ${g['sold_price'].median():.0f}")

print("\nRemoving outliers (>3x median)...")
df_clean = price_model.remove_outliers(df_raw)
print(f"\nAfter cleanup: {len(df_clean)} records ({len(df_raw) - len(df_clean)} removed)")

Raw data: 205 records, 5 items
Date range: 2025-11-10 ~ 2026-05-09

Records per item:
  Helmut Lang SS99                     30 records, median $175
  Number Nine AW03                     53 records, median $185
  Number Nine AW09                     50 records, median $185
  Prada bowling shirt                  29 records, median $375
  Vetements oversized hoodie           43 records, median $300

Removing outliers (>3x median)...
  Helmut Lang SS99: removed 3 outliers (median=$175)
  Number Nine AW03: removed 4 outliers (median=$185)
  Number Nine AW09: removed 4 outliers (median=$185)
  Vetements oversized hoodie: removed 2 outliers (median=$300)

After cleanup: 192 records (13 removed)


## 2. Feature Engineering

We extract features from each transaction:

| Feature | Source | Rationale |
|---------|--------|-----------|
| `week_of_year` | sold_date | Captures seasonal demand cycles |
| `month` | sold_date | Monthly trends (holiday spikes, etc.) |
| `day_of_week` | sold_date | Weekend vs weekday buying patterns |
| `days_since_start` | sold_date | Linear time trend |
| `condition_code` | condition | New (4) → Worn (1); better condition = higher price |
| `followers` | followers | Listing hype / demand signal |
| `item_type_code` | title | **NEW** — Jacket (6) → Shoes (1); captures price tiers within the same keyword |
| `rolling_avg_7/14/30d` | sold_price history | Recent price momentum — the most predictive features |

### Why item_type matters

A single keyword like "Number Nine AW03" can match jackets ($500-800), jeans ($350-400), and t-shirts ($80-100). Without distinguishing item type, the model sees huge variance that isn't noise — it's a missing feature. We classify by matching title keywords against patterns (jacket, pants, hoodie, tee, shirt, shoes).

In [3]:
print("Building features (rolling averages take a moment)...")
df = price_model.build_features(df_clean)

# Show feature sample
display(df[["keyword", "sold_date", "sold_price", "item_type", "condition_code", "followers",
            "week_of_year", "month", "day_of_week", "days_since_start",
            "item_type_code", "rolling_avg_7d", "rolling_avg_14d", "rolling_avg_30d"]].head(10))

print(f"\nFeature matrix shape: {df[price_model.FEATURE_COLS].shape}")
print(f"Features: {price_model.FEATURE_COLS}")

# Item type distribution
print("\n--- Item Type Distribution ---")
type_stats = df.groupby(["keyword", "item_type"]).agg(
    count=("sold_price", "size"),
    median_price=("sold_price", "median"),
).reset_index()
for kw in price_model.TARGET_KEYWORDS:
    kw_data = type_stats[type_stats["keyword"] == kw]
    if kw_data.empty:
        continue
    print(f"\n  {kw}:")
    for _, row in kw_data.sort_values("median_price", ascending=False).iterrows():
        print(f"    {row['item_type']:<10} {row['count']:>2} records  median ${row['median_price']:.0f}")

Building features (rolling averages take a moment)...


,keyword,sold_date,sold_price,item_type,condition_code,followers,week_of_year,month,day_of_week,days_since_start,item_type_code,rolling_avg_7d,rolling_avg_14d,rolling_avg_30d
0,Helmut Lang SS99,2025-11-10,210,pants,2,48,46,11,0,0,5,170.0,170.000000,170.00
1,Helmut Lang SS99,2025-11-12,200,jacket,3,7,46,11,2,2,6,210.0,210.000000,210.00
2,Helmut Lang SS99,2025-11-21,100,tee,3,8,47,11,4,11,2,170.0,205.000000,205.00
3,Helmut Lang SS99,2025-11-24,110,pants,2,77,48,11,0,14,5,100.0,170.000000,170.00
4,Helmut Lang SS99,2025-11-26,150,pants,2,45,48,11,2,16,5,105.0,136.666667,155.00
5,Helmut Lang SS99,2025-12-11,165,shirt,3,15,50,12,3,31,3,170.0,170.000000,140.00
6,Helmut Lang SS99,2025-12-14,120,other,4,86,50,12,6,34,0,165.0,165.000000,131.25
7,Helmut Lang SS99,2025-12-21,150,pants,2,96,51,12,6,41,5,120.0,142.500000,129.00
8,Helmut Lang SS99,2025-12-31,300,pants,3,55,1,12,2,51,5,170.0,150.000000,145.00
9,Helmut Lang SS99,2026-01-03,100,pants,2,47,1,1,5,54,5,300.0,225.000000,183.75



Feature matrix shape: (192, 10)
Features: ['week_of_year', 'month', 'day_of_week', 'days_since_start', 'condition_code', 'followers', 'item_type_code', 'rolling_avg_7d', 'rolling_avg_14d', 'rolling_avg_30d']

--- Item Type Distribution ---

  Number Nine AW03:
    hoodie      3 records  median $315
    jacket      4 records  median $228
    pants      15 records  median $225
    tee        14 records  median $156
    shirt       6 records  median $148
    other       7 records  median $145

  Number Nine AW09:
    hoodie      3 records  median $315
    jacket      4 records  median $228
    pants      13 records  median $225
    tee        13 records  median $168
    shirt       6 records  median $148
    other       7 records  median $145

  Vetements oversized hoodie:
    hoodie     39 records  median $300
    other       2 records  median $186

  Helmut Lang SS99:
    tee         3 records  median $405
    jacket      5 records  median $380
    pants      12 records  median $172
  

## 3. Model Training & Evaluation

We use an **80/20 time-series split** (not random split — this respects temporal ordering and prevents future data leakage). Two models are compared:

- **Linear Regression**: Simple baseline. Assumes linear relationship between features and price.
- **XGBoost** (conservative): Gradient-boosted trees with regularization to prevent overfitting on small datasets.
  - `max_depth=3` (was 4), `n_estimators=50` (was 100), `min_child_weight=5` (new)

**Cross-validation**: 3-fold `TimeSeriesSplit` on the training set to estimate generalization error. If the CV MAE is much higher than the test MAE, the model may be overfitting to the specific test split.

Metrics:
- **MAE** (Mean Absolute Error): Average dollar error — interpretable as "the model is off by $X on average"
- **RMSE** (Root Mean Squared Error): Penalizes large errors more heavily
- **CV MAE**: Cross-validated MAE on training data — better estimate of true generalization

In [4]:
print("Training models per item...\n")
results = price_model.train_and_evaluate(df)

Training models per item...

  Number Nine AW03: LR MAE=$101, XGB MAE=$83, LR CV=$834, XGB CV=$78 | Predicted=$204, 30d Avg=$211, Trend=Stable


  Number Nine AW09: LR MAE=$103, XGB MAE=$83, LR CV=$139, XGB CV=$83 | Predicted=$200, 30d Avg=$211, Trend=Declining
  Vetements oversized hoodie: LR MAE=$137, XGB MAE=$140, LR CV=$302, XGB CV=$116 | Predicted=$371, 30d Avg=$370, Trend=Stable


  Helmut Lang SS99: LR MAE=$375, XGB MAE=$130, LR CV=$626, XGB CV=$105 | Predicted=$273, 30d Avg=$264, Trend=Stable
  Prada bowling shirt: LR MAE=$169, XGB MAE=$136, LR CV=$680, XGB CV=$115 | Predicted=$468, 30d Avg=$380, Trend=Rising


## 4. Visualization — Actual vs Predicted

Each chart shows:
- **Blue dots**: Actual sold prices
- **Red line**: XGBoost predicted trend (or Linear Regression if XGBoost unavailable)
- **Gray dashed line**: Linear Regression baseline
- **Green band**: Train/test split point

In [5]:
for r in results:
    kw = r["keyword"]
    dates = pd.to_datetime(r["dates"])
    actuals = r["actuals"]
    lr_pred = r["lr_pred_all"]
    split_idx = r["train_size"]
    split_date = dates[split_idx]

    fig = go.Figure()

    # Actual prices (blue dots)
    fig.add_trace(go.Scatter(
        x=dates, y=actuals, mode="markers",
        name="Actual Sold Price",
        marker=dict(color="#2563eb", size=7, opacity=0.7),
    ))

    # LR baseline (gray dashed)
    fig.add_trace(go.Scatter(
        x=dates, y=lr_pred, mode="lines",
        name=f"Linear Regression (MAE=${r['lr_mae']:.0f})",
        line=dict(color="#9ca3af", width=2, dash="dash"),
    ))

    # XGBoost (red solid)
    if "xgb_pred_all" in r:
        fig.add_trace(go.Scatter(
            x=dates, y=r["xgb_pred_all"], mode="lines",
            name=f"XGBoost (MAE=${r['xgb_mae']:.0f})",
            line=dict(color="#dc2626", width=2),
        ))

    # Train/test split line (use add_shape to avoid plotly vline bug with dates)
    fig.add_shape(
        type="line",
        x0=split_date, x1=split_date, y0=0, y1=1,
        yref="paper", line=dict(color="#059669", width=1.5, dash="dot"),
    )
    fig.add_annotation(
        x=split_date, y=1.05, yref="paper",
        text="Train | Test", showarrow=False,
        font=dict(color="#059669", size=10),
    )

    trend_emoji = {"Rising": "^", "Declining": "v", "Stable": "-"}[r["trend"]]

    fig.update_layout(
        title=f"{kw}  —  Predicted: ${r['predicted_price']:.0f}  {trend_emoji} {r['trend']} ({r['pct_change']:+.1f}%)",
        xaxis_title="Date", yaxis_title="Price (USD)",
        height=380,
        margin=dict(l=0, r=20, t=50, b=20),
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
    )
    fig.show()

## 5. Model Comparison — MAE Bar Chart

Side-by-side comparison of Linear Regression vs XGBoost MAE per item. Lower is better.

In [6]:
items = [r["keyword"] for r in results]
lr_maes = [r["lr_mae"] for r in results]

fig = go.Figure()
fig.add_trace(go.Bar(
    x=items, y=lr_maes, name="Linear Regression",
    marker_color="#9ca3af", text=[f"${v:.0f}" for v in lr_maes], textposition="outside",
))

if "xgb_mae" in results[0]:
    xgb_maes = [r["xgb_mae"] for r in results]
    fig.add_trace(go.Bar(
        x=items, y=xgb_maes, name="XGBoost",
        marker_color="#dc2626", text=[f"${v:.0f}" for v in xgb_maes], textposition="outside",
    ))

fig.update_layout(
    title="Model Comparison — MAE per Item (lower is better)",
    yaxis_title="MAE ($)",
    barmode="group",
    height=400,
    margin=dict(l=0, r=20, t=40, b=20),
)
fig.show()

## 6. Summary Table

Final output: predicted price, 30-day average, trend direction, and cross-validation scores for each item.

- **Rising**: Predicted price > 30-day avg by more than 5%
- **Declining**: Predicted price < 30-day avg by more than 5%
- **Stable**: Within ±5% of 30-day average
- **CV MAE**: 3-fold time-series cross-validation — a more robust estimate of model error than a single test split

In [7]:
summary = price_model.build_summary_table(results)
display(summary)

# Per-type predictions
display(Markdown("---\n### Predicted Price by Item Type\n"))
for r in results:
    display(Markdown(f"**{r['keyword']}** — Weighted Avg: ${r['predicted_price']:.0f}"))
    tp = r.get("type_predictions", {})
    type_order = ["jacket", "pants", "hoodie", "shirt", "tee", "shoes", "other"]
    lines = []
    for itype in type_order:
        if itype not in tp:
            continue
        info = tp[itype]
        lines.append(
            f"| {itype.capitalize()} | {info['count']} | "
            f"${info['predicted']:.0f} | ${info['median']:.0f} | "
            f"${info['min']:.0f} – ${info['max']:.0f} |"
        )
    if lines:
        header = "| Type | Records | Predicted | Median | Range |\n|---|---|---|---|---|"
        display(Markdown(header + "\n" + "\n".join(lines)))

# Trend interpretation
display(Markdown("---\n### Trend Interpretation\n"))
for r in results:
    emoji = {"Rising": "📈", "Declining": "📉", "Stable": "➡️"}[r["trend"]]
    best_mae = min(r["lr_mae"], r.get("xgb_mae", r["lr_mae"]))
    cv_mae = r.get("xgb_cv_mae", r.get("lr_cv_mae", best_mae))
    display(Markdown(
        f"- **{r['keyword']}** {emoji} {r['trend']} ({r['pct_change']:+.1f}%) — "
        f"Predicted **${r['predicted_price']:.0f}** vs 30-day avg ${r['avg_30d']:.0f}  \n"
        f"  Test MAE: ${best_mae:.0f} | CV MAE: ${cv_mae:.0f}"
    ))

,Item,Records,LR MAE ($),LR RMSE ($),LR CV MAE ($),XGB MAE ($),XGB RMSE ($),XGB CV MAE ($),Predicted Price ($),30-Day Avg ($),Trend,Change (%)
0,Number Nine AW03,49,100.9,124.7,833.6,82.6,100.4,77.8,204.0,211.0,Stable,-3.2
1,Number Nine AW09,46,103.5,126.9,138.9,82.6,106.7,82.7,200.0,211.0,Declining,-5.3
2,Vetements oversized hoodie,41,137.0,183.4,301.7,140.1,203.2,115.8,371.0,370.0,Stable,0.3
3,Helmut Lang SS99,27,375.1,401.7,626.5,130.4,151.0,105.3,273.0,264.0,Stable,3.3
4,Prada bowling shirt,29,169.2,216.7,680.5,135.5,155.1,114.6,468.0,380.0,Rising,23.1


---
### Predicted Price by Item Type


**Number Nine AW03** — Weighted Avg: $204

| Type | Records | Predicted | Median | Range |
|---|---|---|---|---|
| Jacket | 4 | $258 | $228 | $120 – $350 |
| Pants | 15 | $258 | $225 | $108 – $385 |
| Hoodie | 3 | $258 | $315 | $185 – $350 |
| Shirt | 6 | $161 | $148 | $80 – $270 |
| Tee | 14 | $161 | $156 | $90 – $385 |
| Other | 7 | $161 | $145 | $85 – $300 |

**Number Nine AW09** — Weighted Avg: $200

...

- **Number Nine AW03** ➡️ Stable (-3.2%) — Predicted **$204** vs 30-day avg $211  
  Test MAE: $83 | CV MAE: $78

- **Number Nine AW09** 📉 Declining (-5.3%) — Predicted **$200** vs 30-day avg $211  
  Test MAE: $83 | CV MAE: $83

- **Vetements oversized hoodie** ➡️ Stable (+0.3%) — Predicted **$371** vs 30-day avg $370  
  Test MAE: $137 | CV MAE: $116

- **Helmut Lang SS99** ➡️ Stable (+3.3%) — Predicted **$273** vs 30-day avg $264  
  Test MAE: $130 | CV MAE: $105

- **Prada bowling shirt** 📈 Rising (+23.1%) — Predicted **$468** vs 30-day avg $380  
  Test MAE: $136 | CV MAE: $115